# W03 除錯挑戰｜這份 notebook 完全跑得起來，但它有三個問題

**人工智慧於醫學影像組學的分析與應用（3010040）**

---

### 前提

它執行完全正常，**不會噴任何錯誤**。所以不要找語法錯誤。

### 提示

三個問題分別藏在 **「版本」「路徑」「隨機」** 這三件事附近。

### 你要做的事

1. **個人檢視（5 分鐘）**　照提示找，並寫出「**會造成什麼後果**」。
   找到一個就算數。
2. **配對比對（5 分鐘）**　與夥伴比對清單，針對不一致的項目說服對方。
3. **修補（4 分鐘）**　兩人合作實際改寫，改完要能通過這個判準：
   > 把它交給另一組，對方**不問你任何問題**就跑得起來，而且跑得出**一樣的數字**。

把你找到的寫在這裡（雙擊這個儲存格就能編輯）：

| # | 問題在哪一格 | 會造成什麼後果 |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

## 1. 安裝

In [ ]:
# 安裝需要的套件
!pip install -q scikit-learn pandas numpy matplotlib

## 2. 匯入與設定輸出位置

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 輸出位置
OUT_DIR = "/content/w03_output"
os.makedirs(OUT_DIR, exist_ok=True)
print("輸出會存到：", OUT_DIR)

## 3. 載入資料

In [ ]:
# 模擬「120 例已萃取好特徵的 CT 影像資料」
# （為了讓這本 notebook 在任何機器上都跑得動，這裡用合成資料代替真實特徵表）
X, y = make_classification(
    n_samples=120, n_features=30, n_informative=4, n_redundant=5,
    class_sep=1.0, weights=[0.65, 0.35], random_state=2026,
)
feat_names = [f"feature_{i:02d}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feat_names)
df["label"] = y
df.insert(0, "subject", [f"SUB{i+1:03d}" for i in range(len(df))])
print(df.shape)
df.head()

## 4. 切分

In [ ]:
# 切分訓練 / 測試
X_train, X_test, y_train, y_test = train_test_split(
    df[feat_names], df["label"], test_size=0.3, stratify=df["label"]
)
print("train", X_train.shape, "test", X_test.shape)

## 5. 建模與評估

In [ ]:
# 建模與評估
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"測試集 AUC = {auc:.3f}")

## 6. 輸出

In [ ]:
# 輸出結果
result = pd.DataFrame({"metric": ["AUC"], "value": [auc]})
result.to_csv(os.path.join(OUT_DIR, "result.csv"), index=False)

imp = pd.DataFrame({"feature": feat_names, "importance": model.feature_importances_})
imp.sort_values("importance", ascending=False).to_csv(
    os.path.join(OUT_DIR, "feature_importance.csv"), index=False)

print("已輸出：", os.listdir(OUT_DIR))

---
### 還沒頭緒的話，試一件事

從最上面**把整本再跑一次**，把兩次的 AUC 都記下來。

| 第幾次 | AUC |
|---|---|
| 第一次 |  |
| 第二次 |  |

如果兩次不一樣 —— 為什麼？同一份資料、同一個模型、同一段程式。